In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSV files (reusing variable names for efficiency)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

def summarize_df(df, feature_col=None):
    # Basic counts
    row_count = len(df)
    # Convert timestamp to timezone-aware datetimes (UTC -> Asia/Shanghai)
    if 'timestamp' in df.columns and row_count > 0:
        ts = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        min_ts = ts.min()
        max_ts = ts.max()
        # Represent timestamps as ISO strings (compact)
        min_ts_str = None if pd.isna(min_ts) else min_ts.isoformat()
        max_ts_str = None if pd.isna(max_ts) else max_ts.isoformat()
    else:
        min_ts_str = None
        max_ts_str = None

    # Sorted unique cmdb_id (limit to first 50)
    if 'cmdb_id' in df.columns:
        cmdb_vals = sorted(pd.Index(df['cmdb_id'].dropna().unique()).astype(str))[:50]
        cmdb_series = pd.Series(cmdb_vals, name='cmdb_id').astype(str)
    else:
        cmdb_series = pd.Series([], name='cmdb_id').astype(str)

    # Feature/KPI names if provided (limit to first 50)
    if feature_col and feature_col in df.columns:
        feat_vals = sorted(pd.Index(df[feature_col].dropna().unique()).astype(str))[:50]
        feat_series = pd.Series(feat_vals, name=feature_col).astype(str)
    else:
        feat_series = pd.Series([], name=feature_col if feature_col else 'feature').astype(str)

    # Sample up to first 20 rows (include all columns) - keep as DataFrame
    sample = df.head(20).copy()

    summary = {
        'row_count': row_count,
        'min_timestamp': min_ts_str,
        'max_timestamp': max_ts_str,
        'cmdb_ids_sample': cmdb_series,      # pandas Series (compact)
        'feature_names_sample': feat_series, # pandas Series (compact)
        'sample_rows': sample                 # DataFrame with up to 20 rows
    }
    return summary

summaries = {
    'metric.csv': summarize_df(df_metric, feature_col='kpi_name'),
    'trace.csv': summarize_df(df_trace, feature_col='trace_name'),
    'log.csv': summarize_df(df_log, feature_col='log_name'),
    'error_logs.csv': summarize_df(df_error, feature_col=None)
}

# Display the summaries dictionary (compact: Series and DataFrames included)
summaries

```
Out[1]:
```
```python
# Summarize the previous execution results in plain English (compact)
# Reuse the existing 'summaries' variable produced earlier.

parts = []
for fname in ['metric.csv', 'trace.csv', 'log.csv', 'error_logs.csv']:
    info = summaries[fname]
    row_count = info['row_count']
    min_ts = info['min_timestamp']
    max_ts = info['max_timestamp']
    cmdb_series = info['cmdb_ids_sample']
    feat_series = info['feature_names_sample']
    # compact samples: show up to first 10 items
    cmdb_sample = ", ".join(cmdb_series.astype(str).tolist()[:10]) or "none"
    feat_sample = ", ".join(feat_series.astype(str).tolist()[:10]) or "none"
    sample_rows_count = info['sample_rows'].shape[0]
    parts.append(
        f"{fname}: {row_count} rows; time range {min_ts} to {max_ts} (UTC+8); "
        f"cmdb_ids (first up to 10): {cmdb_sample}; "
        f"feature/KPI names (first up to 10): {feat_sample}; "
        f"sample rows shown: {sample_rows_count}"
    )

summary = "Summary of loaded CSV files (straightforward):\n" + "\n".join(parts)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'row_count': 149772, 'min_timestamp': '2022-03-21T14:30:00+08:00', 'max_timestamp': '2022-03-21T14:59:00+08:00', 'cmdb_ids_sample': 0                    adservice
1                  adservice-0
2                  adservice-1
3                  adservice-2
4                   adservice2
                ...           
45     productcatalogservice-2
46    productcatalogservice2-0
47       recommendationservice
48     recommendationservice-0
49     recommendationservice-1
Name: cmdb_id, Length: 50, dtype: object, 'feature_names_sample': 0                                        app.grpc.count
1                                          app.grpc.mrt
2                                           app.grpc.rr
3                                           app.grpc.sr
4                                        app.http.count
                            ...                        
45               container.node-5.container_memory_swap
46           container.node-5.container_memory_usage_MB
47     container.node-5.container_memory_working_set_MB
48    container.node-5.container_network_receive_MB....
49    container.node-5.container_network_receive_err...
Name: kpi_name, Length: 50, dtype: object, 'sample_rows':      timestamp    cmdb_id                                           kpi_name         value
0   1647844200  adservice                                     app.grpc.count    232.000000
1   1647844200  adservice                                       app.grpc.mrt      0.002325
2   1647844200  adservice                                        app.grpc.rr    100.000000
3   1647844200  adservice                                        app.grpc.sr     78.879310
4   1647844200  adservice                                     app.http.count     24.000000
5   1647844200  adservice                                       app.http.mrt      0.008750
6   1647844200  adservice                                        app.http.rr    100.000000
7   1647844200  adservice                                        app.http.sr    100.000000
8   1647844200  adservice    runtime.java_lang_ClassLoading_LoadedClassCount   4990.000000
9   1647844200  adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...   4990.000000
10  1647844200  adservice  runtime.java_lang_ClassLoading_UnloadedClassCount      0.000000
11  1647844200  adservice             runtime.java_lang_ClassLoading_Verbose      0.000000
12  1647844200  adservice  runtime.java_lang_Compilation_CompilationTimeM...      1.000000
13  1647844200  adservice  runtime.java_lang_Compilation_TotalCompilation...  43975.166667
14  1647844200  adservice  runtime.java_lang_GarbageCollector_CollectionC...  47331.000000
15  1647844200  adservice  runtime.java_lang_GarbageCollector_CollectionC...      3.000000
16  1647844200  adservice  runtime.java_lang_GarbageCollector_CollectionT...  74368.416667
17  1647844200  adservice  runtime.java_lang_GarbageCollector_CollectionT...    177.000000
18  1647844200  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...      1.000000
19  1647844200  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...      1.000000}, 'trace.csv': {'row_count': 36048, 'min_timestamp': '2022-03-21T14:30:00+08:00', 'max_timestamp': '2022-03-21T14:59:00+08:00', 'cmdb_ids_sample': 0                  adservice-0
1                  adservice-1
2                  adservice-2
3                 adservice2-0
4                cartservice-0
                ...           
35    recommendationservice2-0
36           shippingservice-0
37           shippingservice-1
38           shippingservice-2
39          shippingservice2-0
Name: cmdb_id, Length: 40, dtype: object, 'feature_names_sample': 0           trace.from_checkoutservice-0.duration_mean
1            trace.from_checkoutservice-0.duration_p95
2              trace.from_checkoutservice-0.error_rate
3               trace.from_checkoutservice-0.row_count
4           trace.from_checkoutservice-1.duration_mean
                            ...                       
45    trace.from_recommendationservice2-0.duration_p95
46      trace.from_recommendationservice2-0.error_rate
47       trace.from_recommendationservice2-0.row_count
48                       trace.from_root.duration_mean
49                        trace.from_root.duration_p95
Name: trace_name, Length: 50, dtype: object, 'sample_rows':      timestamp      cmdb_id                           trace_name      value
0   1647844200  adservice-0  trace.from_frontend-0.duration_mean   0.000012
1   1647844200  adservice-0   trace.from_frontend-0.duration_p95   0.000013
2   1647844200  adservice-0     trace.from_frontend-0.error_rate   0.000000
3   1647844200  adservice-0      trace.from_frontend-0.row_count  27.000000
4   1647844200  adservice-0  trace.from_frontend-1.duration_mean   0.000012
5   1647844200  adservice-0   trace.from_frontend-1.duration_p95   0.000018
6   1647844200  adservice-0     trace.from_frontend-1.error_rate   0.000000
7   1647844200  adservice-0      trace.from_frontend-1.row_count  20.000000
8   1647844200  adservice-0  trace.from_frontend-2.duration_mean   0.000013
9   1647844200  adservice-0   trace.from_frontend-2.duration_p95   0.000017
10  1647844200  adservice-0     trace.from_frontend-2.error_rate   0.000000
11  1647844200  adservice-0      trace.from_frontend-2.row_count   8.000000
12  1647844200  adservice-1  trace.from_frontend-0.duration_mean   0.000012
13  1647844200  adservice-1   trace.from_frontend-0.duration_p95   0.000013
14  1647844200  adservice-1     trace.from_frontend-0.error_rate   0.000000
15  1647844200  adservice-1      trace.from_frontend-0.row_count  26.000000
16  1647844200  adservice-1  trace.from_frontend-1.duration_mean   0.000013
17  1647844200  adservice-1   trace.from_frontend-1.duration_p95   0.000020
18  1647844200  adservice-1     trace.from_frontend-1.error_rate   0.000000
19  1647844200  adservice-1      trace.from_frontend-1.row_count  20.000000}, 'log.csv': {'row_count': 1814, 'min_timestamp': '2022-03-21T14:30:00+08:00', 'max_timestamp': '2022-03-21T14:59:00+08:00', 'cmdb_ids_sample': 0                 adservice-0
1                 adservice-1
2                 adservice-2
3               cartservice-0
4               cartservice-1
               ...           
26    recommendationservice-2
27               redis-cart-0
28          shippingservice-0
29          shippingservice-1
30          shippingservice-2
Name: cmdb_id, Length: 31, dtype: object, 'feature_names_sample': 0    log.error_count
1      log.row_count
Name: log_name, dtype: object, 'sample_rows':      timestamp            cmdb_id         log_name   value
0   1647844200        adservice-0  log.error_count     0.0
1   1647844200        adservice-0    log.row_count  1218.0
2   1647844200        adservice-1  log.error_count     0.0
3   1647844200        adservice-1    log.row_count   110.0
4   1647844200        adservice-2  log.error_count     0.0
5   1647844200        adservice-2    log.row_count   102.0
6   1647844200      cartservice-0  log.error_count     0.0
7   1647844200      cartservice-0    log.row_count   773.0
8   1647844200      cartservice-1  log.error_count     0.0
9   1647844200      cartservice-1    log.row_count   790.0
10  1647844200      cartservice-2  log.error_count     0.0
11  1647844200      cartservice-2    log.row_count   801.0
12  1647844200  checkoutservice-0  log.error_count     0.0
13  1647844200  checkoutservice-0    log.row_count    48.0
14  1647844200  checkoutservice-1  log.error_count     0.0
15  1647844200  checkoutservice-1    log.row_count    51.0
16  1647844200  checkoutservice-2  log.error_count     0.0
17  1647844200  checkoutservice-2    log.row_count    48.0
18  1647844200  currencyservice-0  log.error_count     0.0
19  1647844200  currencyservice-0    log.row_count   670.0}, 'error_logs.csv': {'row_count': 1540, 'min_timestamp': '2022-03-21T14:30:00+08:00', 'max_timestamp': '2022-03-21T14:59:58+08:00', 'cmdb_ids_sample': 0    frontend-0
1    frontend-1
2    frontend-2
Name: cmdb_id, dtype: object, 'feature_names_sample': Series([], Name: feature, dtype: object), 'sample_rows':      timestamp     cmdb_id                                            message
0   1647844200  frontend-0  severity: warning, message: failed to retrieve...
1   1647844201  frontend-1  severity: warning, message: failed to retrieve...
2   1647844201  frontend-2  severity: warning, message: failed to retrieve...
3   1647844202  frontend-0  severity: warning, message: failed to retrieve...
4   1647844204  frontend-0  severity: warning, message: failed to retrieve...
5   1647844205  frontend-1  severity: warning, message: failed to retrieve...
6   1647844206  frontend-2  severity: warning, message: failed to retrieve...
7   1647844207  frontend-0  severity: warning, message: failed to retrieve...
8   1647844209  frontend-0  severity: warning, message: failed to retrieve...
9   1647844210  frontend-1  severity: warning, message: failed to retrieve...
10  1647844212  frontend-1  severity: warning, message: failed to retrieve...
11  1647844212  frontend-2  severity: warning, message: failed to retrieve...
12  1647844214  frontend-0  severity: warning, message: failed to retrieve...
13  1647844214  frontend-1  severity: warning, message: failed to retrieve...
14  1647844215  frontend-0  severity: warning, message: failed to retrieve...
15  1647844216  frontend-1  severity: warning, message: failed to retrieve...
16  1647844218  frontend-0  severity: warning, message: failed to retrieve...
17  1647844219  frontend-1  severity: warning, message: failed to retrieve...
18  1647844219  frontend-2  severity: warning, message: failed to retrieve...
19  1647844221  frontend-0  severity: warning, message: failed to retrieve...}}```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the entire metric.csv
df_p95 = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95)
df_p95 = df_p95.rename(columns={'value': 'global_p95'})

# 2) Filter metric.csv to the incident window (UTC+8)
start_local = tz.localize(datetime(2022, 3, 21, 14, 30, 0))
end_local = tz.localize(datetime(2022, 3, 21, 15, 0, 0))
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts = int(end_local.astimezone(pytz.utc).timestamp())

df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Join windowed rows with global P95, mark anomalies (value > global_p95)
df_win_merged = df_window.merge(df_p95, on=['cmdb_id', 'kpi_name'], how='left')
df_win_merged['is_anom'] = df_win_merged['value'] > df_win_merged['global_p95']

# Aggregations within the window
# a) anomaly counts and earliest anomaly timestamp (in epoch seconds UTC)
if not df_win_merged.empty:
    df_anom = (
        df_win_merged[df_win_merged['is_anom']]
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(earliest_anom_ts=('timestamp', 'min'), anomaly_count=('is_anom', 'sum'))
    )
else:
    df_anom = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'earliest_anom_ts', 'anomaly_count'])

# b) max value in window per group
if not df_window.empty:
    df_max = df_window.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(max_value_in_window=('value', 'max'))
else:
    df_max = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'max_value_in_window'])

# 3 continued) Merge everything with the full list of (cmdb_id, kpi_name) from df_p95
df_stats = df_p95.merge(df_anom, on=['cmdb_id', 'kpi_name'], how='left').merge(df_max, on=['cmdb_id', 'kpi_name'], how='left')

# Fill missing anomaly_count with 0 (groups with no window rows or no anomalies)
df_stats['anomaly_count'] = df_stats['anomaly_count'].fillna(0).astype(int)

# Convert earliest_anom_ts to localized ISO strings (UTC+8); keep NaN as None
def to_local_iso(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).isoformat()

df_stats['earliest_anomaly_timestamp'] = df_stats['earliest_anom_ts'].apply(to_local_iso)
df_stats = df_stats.drop(columns=['earliest_anom_ts'])

# 4) Prepare final ranking: top 20 by anomaly_count desc, tie-breaker by max_value_in_window desc
# If all anomaly_count == 0, instead rank by max_value_in_window desc
if (df_stats['anomaly_count'] == 0).all():
    df_ranked = df_stats.sort_values(by=['max_value_in_window'], ascending=[False])
else:
    df_ranked = df_stats.sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False])

# Select and order required columns, keep top 20
result_top20 = df_ranked[['cmdb_id', 'kpi_name', 'global_p95', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']].head(20)

# Keep intermediate variables in kernel for follow-ups:
# - df_p95 (global thresholds)
# - df_window (filtered rows in incident window)
# - df_win_merged (window rows joined with global_p95 and anomaly flag)
# - df_stats (aggregated stats per (cmdb_id, kpi_name))
df_p95, df_window.shape, df_win_merged.shape, df_stats.shape, result_top20

```
Out[2]:
```
```python
# Build a compact plain-English summary of the previously computed top-20 results.
# Reuse result_top20 (created in prior step) and df_stats for overall counts.

top = result_top20.copy()
total_groups_with_anom = int((df_stats['anomaly_count'] > 0).sum())
total_groups = int(df_stats.shape[0])

lines = []
header = f"Metric anomaly summary (window 2022-03-21 14:30 to 15:00 UTC+8): {total_groups_with_anom} of {total_groups} (cmdb_id,kpi_name) had value > global P95 at least once."
lines.append(header)
lines.append("Top candidates (up to 20) ranked by anomaly count:")

for i, row in top.reset_index(drop=True).iterrows():
    idx = i + 1
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    p95 = row['global_p95']
    cnt = int(row['anomaly_count'])
    earliest = row['earliest_anomaly_timestamp'] if row['earliest_anomaly_timestamp'] is not None else "N/A"
    maxv = row['max_value_in_window'] if not pd.isna(row['max_value_in_window']) else "N/A"
    lines.append(f"{idx}. {cmdb} | {kpi} — anomalies={cnt}, earliest={earliest}, max={maxv}, global_P95={p95}")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name    global_p95
0              adservice                                     app.grpc.count    252.300000
1              adservice                                       app.grpc.mrt      0.002448
2              adservice                                        app.grpc.rr    100.000000
3              adservice                                        app.grpc.sr     79.419163
4              adservice                                     app.http.count     24.000000
...                  ...                                                ...           ...
4997  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...     33.050000
4998  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....  86970.000000
4999  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      0.001050
5000  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     25.000000
5001  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....   2537.250000

[5002 rows x 3 columns], (149772, 4), (149772, 6), (5002, 6),          cmdb_id                                           kpi_name    global_p95  anomaly_count earliest_anomaly_timestamp  max_value_in_window
244    adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  1.019248e+12              2  2022-03-21T14:58:00+08:00         1.019515e+12
835   adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime  1.254475e+11              2  2022-03-21T14:58:00+08:00         1.257608e+11
253    adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  9.923126e+10              2  2022-03-21T14:57:00+08:00         9.928187e+10
255    adservice  runtime.java_lang_Threading_CurrentThreadUserTime  9.654046e+10              2  2022-03-21T14:57:00+08:00         9.660417e+10
252    adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  7.870851e+10              2  2022-03-21T14:58:00+08:00         7.873011e+10
844   adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime  8.303044e+09              2  2022-03-21T14:58:00+08:00         8.324734e+09
846   adservice2  runtime.java_lang_Threading_CurrentThreadUserTime  7.742708e+09              2  2022-03-21T14:58:00+08:00         7.760833e+09
3423      node-4                             system.fs.inodes.total  6.509554e+09              2  2022-03-21T14:30:00+08:00         6.510090e+09
3421      node-4                              system.fs.inodes.free  6.506961e+09              2  2022-03-21T14:30:00+08:00         6.507497e+09
3420      node-4                                   system.disk.used  5.794167e+09              2  2022-03-21T14:58:00+08:00         5.794376e+09
843   adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...  5.716434e+09              2  2022-03-21T14:58:00+08:00         5.737365e+09
3364      node-3                             system.fs.inodes.total  4.891692e+09              2  2022-03-21T14:30:00+08:00         4.892454e+09
3306      node-2                             system.fs.inodes.total  4.891935e+09              2  2022-03-21T14:30:00+08:00         4.892355e+09
3247      node-1                             system.fs.inodes.total  4.891935e+09              2  2022-03-21T14:30:00+08:00         4.892349e+09
3362      node-3                              system.fs.inodes.free  4.889741e+09              2  2022-03-21T14:30:00+08:00         4.890503e+09
3245      node-1                              system.fs.inodes.free  4.889991e+09              2  2022-03-21T14:30:00+08:00         4.890406e+09
3304      node-2                              system.fs.inodes.free  4.889962e+09              2  2022-03-21T14:30:00+08:00         4.890382e+09
3361      node-3                                   system.disk.used  4.354767e+09              2  2022-03-21T14:58:00+08:00         4.354976e+09
3244      node-1                                   system.disk.used  4.350587e+09              2  2022-03-21T14:58:00+08:00         4.350758e+09
3303      node-2                                   system.disk.used  4.350192e+09              2  2022-03-21T14:58:00+08:00         4.350361e+09)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse df_stats from previous steps (must exist in the kernel)
try:
    df_stats
except NameError:
    raise RuntimeError("df_stats not found in kernel. Run previous aggregation step first.")

tz = pytz.timezone('Asia/Shanghai')

# Parse earliest_anomaly_timestamp into timezone-aware datetimes (NaT for missing)
dt_series = pd.to_datetime(df_stats['earliest_anomaly_timestamp'], utc=False, errors='coerce')
# Ensure timezone is Asia/Shanghai for all entries that are timezone-naive or already tz-aware
# If dt is tz-naive, localize; if tz-aware, convert to Asia/Shanghai
def ensure_shanghai(ts):
    if pd.isna(ts):
        return pd.NaT
    if ts.tzinfo is None:
        return tz.localize(ts)
    return ts.tz_convert(tz)

dt_series = dt_series.apply(ensure_shanghai)
df_stats = df_stats.assign(_earliest_dt=dt_series)

# Helper to produce top_kpis string per group
def top_kpis_list(subdf, top_n=5):
    # sort by anomaly_count desc, then by max_value_in_window desc as secondary
    s = subdf.sort_values(['anomaly_count', 'max_value_in_window'], ascending=[False, False])
    pairs = [f"{k}:{int(a)}" for k, a in zip(s['kpi_name'].tolist(), s['anomaly_count'].tolist())][:top_n]
    return "; ".join(pairs)

# Aggregate per cmdb_id
grouped = df_stats.groupby('cmdb_id', sort=True)

agg_list = []
for cmdb, g in grouped:
    num_kpis_with_anom = int((g['anomaly_count'] > 0).sum())
    total_anomaly_points = int(g['anomaly_count'].sum())
    # earliest anomaly timestamp across kpis (ignore NaT)
    if g['_earliest_dt'].dropna().empty:
        earliest_iso = None
    else:
        earliest_dt = g['_earliest_dt'].dropna().min()
        earliest_iso = earliest_dt.isoformat()
    top_kpis = top_kpis_list(g, top_n=5)
    agg_list.append((cmdb, num_kpis_with_anom, total_anomaly_points, earliest_iso, top_kpis))

df_component_agg = pd.DataFrame(
    agg_list,
    columns=['cmdb_id', 'num_kpis_with_anom', 'total_anomaly_points', 'earliest_anomaly_timestamp', 'top_kpis']
)

# Sort by total_anomaly_points descending, then by num_kpis_with_anom desc
df_component_agg = df_component_agg.sort_values(by=['total_anomaly_points', 'num_kpis_with_anom'], ascending=[False, False])

# Keep only top 20 rows for compact output
df_component_agg_top20 = df_component_agg.head(20)

# Expose df_component_agg (full aggregated table) and compact top-20 view for follow-ups
df_component_agg, df_component_agg_top20

```
Out[3]:
```
```python
# Summarize the component-level aggregation results in plain English.
# Reuse df_component_agg and df_component_agg_top20 from the kernel.

try:
    df_component_agg
    df_component_agg_top20
except NameError:
    raise RuntimeError("Expected df_component_agg and df_component_agg_top20 in kernel. Run previous steps first.")

# Basic counts
total_components = int(df_component_agg.shape[0])
components_with_anom = int((df_component_agg['total_anomaly_points'] > 0).sum())

# Top 5 components by total_anomaly_points for a brief highlight
top5 = df_component_agg.sort_values(['total_anomaly_points', 'num_kpis_with_anom'], ascending=[False, False]).head(5)

lines = []
lines.append(f"Component-level anomaly summary (window 2022-03-21 14:30–15:00 UTC+8):")
lines.append(f"- Components analyzed: {total_components}")
lines.append(f"- Components with anomalies (total_anomaly_points>0): {components_with_anom}")
lines.append("")
lines.append("Top 5 components by total anomaly points:")

for i, row in top5.reset_index(drop=True).iterrows():
    cmdb = row['cmdb_id']
    total_pts = int(row['total_anomaly_points'])
    nkpis = int(row['num_kpis_with_anom'])
    earliest = row['earliest_anomaly_timestamp'] if row['earliest_anomaly_timestamp'] is not None else "N/A"
    top_kpis = row['top_kpis'] if isinstance(row['top_kpis'], str) else ""
    # keep top_kpis short for readability
    if len(top_kpis) > 120:
        top_kpis = top_kpis[:117] + "..."
    lines.append(f"{i+1}. {cmdb} — total_anomaly_points={total_pts}, num_kpis_with_anom={nkpis}, earliest={earliest}")
    lines.append(f"    top_kpis: {top_kpis}")

lines.append("")
lines.append("Top-20 compact table is available in variable `df_component_agg_top20` for inspection.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id  num_kpis_with_anom  total_anomaly_points earliest_anomaly_timestamp                                           top_kpis
30            frontend2-0                  80                   146  2022-03-21T14:30:00+08:00  container.node-6.container_last_seen:2; contai...
27             frontend-0                  80                   144  2022-03-21T14:31:00+08:00  container.node-5.container_last_seen:2; contai...
29             frontend-2                  80                   139  2022-03-21T14:32:00+08:00  container.node-5.container_last_seen:2; contai...
28             frontend-1                  78                   134  2022-03-21T14:31:00+08:00  container.node-5.container_last_seen:2; contai...
4              adservice2                  50                    90  2022-03-21T14:30:00+08:00  runtime.java_lang_OperatingSystem_ProcessCpuTi...
..                    ...                 ...                   ...                        ...                                                ...
42  productcatalogservice                   2                     4  2022-03-21T14:40:00+08:00  app.grpc.count:2; app.grpc.mrt:2; app.grpc.rr:...
47  recommendationservice                   2                     4  2022-03-21T14:38:00+08:00  app.grpc.count:2; app.grpc.mrt:2; app.grpc.rr:...
54        shippingservice                   2                     4  2022-03-21T14:55:00+08:00  app.grpc.count:2; app.grpc.mrt:2; app.grpc.rr:...
21           emailservice                   2                     3  2022-03-21T14:54:00+08:00  app.grpc.mrt:2; app.grpc.count:1; app.grpc.rr:...
37         paymentservice                   2                     3  2022-03-21T14:47:00+08:00  app.grpc.mrt:2; app.grpc.count:1; app.grpc.rr:...

[59 rows x 5 columns],                      cmdb_id  num_kpis_with_anom  total_anomaly_points earliest_anomaly_timestamp                                           top_kpis
30               frontend2-0                  80                   146  2022-03-21T14:30:00+08:00  container.node-6.container_last_seen:2; contai...
27                frontend-0                  80                   144  2022-03-21T14:31:00+08:00  container.node-5.container_last_seen:2; contai...
29                frontend-2                  80                   139  2022-03-21T14:32:00+08:00  container.node-5.container_last_seen:2; contai...
28                frontend-1                  78                   134  2022-03-21T14:31:00+08:00  container.node-5.container_last_seen:2; contai...
4                 adservice2                  50                    90  2022-03-21T14:30:00+08:00  runtime.java_lang_OperatingSystem_ProcessCpuTi...
0                  adservice                  46                    90  2022-03-21T14:30:00+08:00  runtime.java_lang_OperatingSystem_ProcessCpuTi...
46  productcatalogservice2-0                  47                    88  2022-03-21T14:31:00+08:00  container.node-6.container_last_seen:2; contai...
14         checkoutservice-2                  62                    87  2022-03-21T14:30:00+08:00  container.node-5.container_last_seen:2; contai...
43   productcatalogservice-0                  48                    87  2022-03-21T14:31:00+08:00  container.node-5.container_last_seen:2; contai...
45   productcatalogservice-2                  46                    83  2022-03-21T14:30:00+08:00  container.node-5.container_last_seen:2; contai...
15        checkoutservice2-0                  58                    81  2022-03-21T14:31:00+08:00  container.node-5.container_last_seen:2; contai...
44   productcatalogservice-1                  45                    81  2022-03-21T14:32:00+08:00  container.node-5.container_last_seen:2; contai...
12         checkoutservice-0                  40                    79  2022-03-21T14:32:00+08:00  container.node-5.container_last_seen:2; contai...
10            cartservice2-0                  33                    65  2022-03-21T14:30:00+08:00  container.node-6.container_last_seen:2; contai...
36                    node-6                  37                    64  2022-03-21T14:30:00+08:00  system.fs.inodes.total:2; system.fs.inodes.fre...
31                    node-1                  35                    64  2022-03-21T14:30:00+08:00  system.fs.inodes.total:2; system.fs.inodes.fre...
13         checkoutservice-1                  33                    64  2022-03-21T14:30:00+08:00  container.node-5.container_last_seen:2; contai...
32                    node-2                  32                    61  2022-03-21T14:30:00+08:00  system.fs.inodes.total:2; system.fs.inodes.fre...
9              cartservice-2                  32                    60  2022-03-21T14:33:00+08:00  container.node-5.container_last_seen:2; contai...
33                    node-3                  34                    59  2022-03-21T14:30:00+08:00  system.fs.inodes.total:2; system.fs.inodes.fre...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing dataframes if present; otherwise load
try:
    df_error
except NameError:
    df_error = pd.read_csv('error_logs.csv')

try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')

# Define window in UTC epoch seconds (inclusive)
start_local = tz.localize(datetime(2022, 3, 21, 14, 30, 0))
end_local   = tz.localize(datetime(2022, 3, 21, 14, 35, 0))
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts   = int(end_local.astimezone(pytz.utc).timestamp())

# --------------------------
# Part 1: error_logs.csv
# --------------------------
# Filter window
df_error_win = df_error[(df_error['timestamp'] >= start_ts) & (df_error['timestamp'] <= end_ts)].copy()

# a) Top 20 cmdb_id by message count
s_cmdb = df_error_win.groupby('cmdb_id').size().reset_index()
s_cmdb.columns = ['cmdb_id', 'message_count']
df_error_top_cmdb = s_cmdb.sort_values('message_count', ascending=False).head(20).reset_index(drop=True)

# b) Top 10 most frequent message texts (truncate to first 200 chars for display)
df_error_win['message_short'] = df_error_win['message'].fillna('').astype(str).str.slice(0,200)
s_msgs = df_error_win.groupby('message_short').size().reset_index()
s_msgs.columns = ['message_short', 'count']
df_error_top_msgs = s_msgs.sort_values('count', ascending=False).head(10).reset_index(drop=True)

# c) Up to 20 sample rows (timestamp, cmdb_id, message)
df_error_sample = df_error_win[['timestamp', 'cmdb_id', 'message']].head(20).copy().reset_index(drop=True)

# --------------------------
# Part 2: trace.csv
# --------------------------
# Filter window
df_trace_win = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Filter trace_name conditions
pattern_src = r'from_frontend|from_root'
pattern_metric = r'error_rate|row_count|duration_p95|duration_mean'
mask = df_trace_win['trace_name'].str.contains(pattern_src, regex=True, na=False) & df_trace_win['trace_name'].str.contains(pattern_metric, regex=True, na=False)
df_trace_filtered = df_trace_win[mask].copy()

# Group by (cmdb_id, trace_name) and compute avg, max, earliest timestamp where value>0
if not df_trace_filtered.empty:
    df_trace_agg = (
        df_trace_filtered.groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(avg_value=('value', 'mean'), max_value=('value', 'max'))
    )
    df_nonzero = df_trace_filtered[df_trace_filtered['value'] > 0].copy()
    if not df_nonzero.empty:
        df_earliest = (
            df_nonzero.groupby(['cmdb_id', 'trace_name'], as_index=False)
            .agg(earliest_nonzero_ts=('timestamp', 'min'))
        )
    else:
        df_earliest = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'earliest_nonzero_ts'])
    df_trace_grouped = df_trace_agg.merge(df_earliest, on=['cmdb_id', 'trace_name'], how='left')
else:
    df_trace_grouped = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'avg_value', 'max_value', 'earliest_nonzero_ts'])

# Convert earliest_nonzero_ts to localized ISO strings (UTC+8); keep None for missing
def ts_to_local_iso(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).isoformat()

if 'earliest_nonzero_ts' in df_trace_grouped.columns:
    df_trace_grouped['earliest_nonzero_timestamp'] = df_trace_grouped['earliest_nonzero_ts'].apply(ts_to_local_iso)
    df_trace_grouped = df_trace_grouped.drop(columns=['earliest_nonzero_ts'])
else:
    df_trace_grouped['earliest_nonzero_timestamp'] = None

# Top 20 groups sorted by max_value desc
df_trace_top20 = df_trace_grouped.sort_values('max_value', ascending=False).head(20).reset_index(drop=True)

# Keep full intermediate tables in kernel variables for follow-ups:
# df_error_win, df_error_top_cmdb, df_error_top_msgs, df_error_sample,
# df_trace_win, df_trace_filtered, df_trace_grouped, df_trace_top20

# Return compact outputs (up to limits requested)
df_error_top_cmdb, df_error_top_msgs, df_error_sample, df_trace_top20

```
Out[4]:
```
Summary (window 2022-03-21 14:30–14:35 UTC+8)

1) error_logs.csv (14:30–14:35)
- Top components by message count:
  - frontend-0: 129 messages
  - frontend-1: 83 messages
  - frontend-2: 49 messages
- Top message text (truncated to first 200 chars) by frequency:
  - "severity: warning, message: failed to retrieve..." — 261 occurrences
- Sample rows: multiple warning messages from frontend-0/1/2 starting at 2022-03-21T14:30:00+08:00 and continuing through the 14:30–14:35 window.

Interpretation: the frontend pods produced the majority of error log entries in the early incident window, dominated by repeated "failed to retrieve..." warnings.

2) trace.csv (14:30–14:35, only trace_name containing from_frontend or from_root AND one of error_rate/row_count/duration_p95/duration_mean)
- Aggregated top groups (ranked by max_value, top 20 — showing the leading entries):
  1. productcatalogservice2-0 — trace.from_frontend2-0.row_count: avg=526.67, max=633.0, earliest_nonzero=2022-03-21T14:30:00+08:00
  2. currencyservice2-0 — trace.from_frontend2-0.row_count: avg=344.33, max=394.0, earliest_nonzero=2022-03-21T14:30:00+08:00
  3. productcatalogservice-1 — trace.from_frontend-0.row_count: avg=196.33, max=206.0, earliest_nonzero=2022-03-21T14:30:00+08:00
  4. productcatalogservice-2 — trace.from_frontend-0.row_count: avg=196.33, max=206.0, earliest_nonzero=2022-03-21T14:30:00+08:00
  5. productcatalogservice-0 — trace.from_frontend-0.row_count: avg=196.33, max=205.0, earliest_nonzero=2022-03-21T14:30:00+08:00
  6. currencyservice-2 — trace.from_frontend-0.row_count: avg=126.83, max=140.0, earliest_nonzero=2022-03-21T14:30:00+08:00
  7. currencyservice-0 — trace.from_frontend-0.row_count: avg=126.83, max=140.0, earliest_nonzero=2022-03-21T14:30:00+08:00
  8. currencyservice-1 — trace.from_frontend-0.row_count: avg=126.67, max=139.0, earliest_nonzero=2022-03-21T14:30:00+08:00
  9. frontend-0 — trace.from_root.row_count: avg=129.17, max=134.0, earliest_nonzero=2022-03-21T14:30:00+08:00
 10. frontend2-0 — trace.from_root.row_count: avg=115.00, max=131.0, earliest_nonzero=2022-03-21T14:30:00+08:00
- Many of the top groups are row_count metrics for services reached from frontend (productcatalogservice variants, currencyservice variants, etc.). Earliest non-zero occurrences for these top groups are at 14:30:00+08:00.

Interpretation: at the start of the incident window the frontend pods produced many warnings, and downstream services (notably productcatalogservice and currencyservice instances) show high request row_count from frontend (large avg and max row_count values), with activity present from 14:30. This aligns with frontend errors and high traffic/loads to backend services at the same early timestamp.

If you want, I can:
- show more message text examples (full messages) or more sample rows,
- expand the trace top table beyond 20,
- or correlate specific frontend error timestamps with individual trace groups.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  message_count
0  frontend-0            129
1  frontend-1             83
2  frontend-2             49,                                        message_short  count
0  severity: warning, message: failed to retrieve...    261,      timestamp     cmdb_id                                            message
0   1647844200  frontend-0  severity: warning, message: failed to retrieve...
1   1647844201  frontend-1  severity: warning, message: failed to retrieve...
2   1647844201  frontend-2  severity: warning, message: failed to retrieve...
3   1647844202  frontend-0  severity: warning, message: failed to retrieve...
4   1647844204  frontend-0  severity: warning, message: failed to retrieve...
5   1647844205  frontend-1  severity: warning, message: failed to retrieve...
6   1647844206  frontend-2  severity: warning, message: failed to retrieve...
7   1647844207  frontend-0  severity: warning, message: failed to retrieve...
8   1647844209  frontend-0  severity: warning, message: failed to retrieve...
9   1647844210  frontend-1  severity: warning, message: failed to retrieve...
10  1647844212  frontend-1  severity: warning, message: failed to retrieve...
11  1647844212  frontend-2  severity: warning, message: failed to retrieve...
12  1647844214  frontend-0  severity: warning, message: failed to retrieve...
13  1647844214  frontend-1  severity: warning, message: failed to retrieve...
14  1647844215  frontend-0  severity: warning, message: failed to retrieve...
15  1647844216  frontend-1  severity: warning, message: failed to retrieve...
16  1647844218  frontend-0  severity: warning, message: failed to retrieve...
17  1647844219  frontend-1  severity: warning, message: failed to retrieve...
18  1647844219  frontend-2  severity: warning, message: failed to retrieve...
19  1647844221  frontend-0  severity: warning, message: failed to retrieve...,                      cmdb_id                        trace_name   avg_value  max_value earliest_nonzero_timestamp
0   productcatalogservice2-0  trace.from_frontend2-0.row_count  526.666667      633.0  2022-03-21T14:30:00+08:00
1         currencyservice2-0  trace.from_frontend2-0.row_count  344.333333      394.0  2022-03-21T14:30:00+08:00
2    productcatalogservice-1   trace.from_frontend-0.row_count  196.333333      206.0  2022-03-21T14:30:00+08:00
3    productcatalogservice-2   trace.from_frontend-0.row_count  196.333333      206.0  2022-03-21T14:30:00+08:00
4    productcatalogservice-0   trace.from_frontend-0.row_count  196.333333      205.0  2022-03-21T14:30:00+08:00
5          currencyservice-2   trace.from_frontend-0.row_count  126.833333      140.0  2022-03-21T14:30:00+08:00
6          currencyservice-0   trace.from_frontend-0.row_count  126.833333      140.0  2022-03-21T14:30:00+08:00
7          currencyservice-1   trace.from_frontend-0.row_count  126.666667      139.0  2022-03-21T14:30:00+08:00
8                 frontend-0         trace.from_root.row_count  129.166667      134.0  2022-03-21T14:30:00+08:00
9                frontend2-0         trace.from_root.row_count  115.000000      131.0  2022-03-21T14:30:00+08:00
10   productcatalogservice-1   trace.from_frontend-1.row_count  123.333333      130.0  2022-03-21T14:30:00+08:00
11   productcatalogservice-2   trace.from_frontend-1.row_count  123.333333      130.0  2022-03-21T14:30:00+08:00
12   productcatalogservice-0   trace.from_frontend-1.row_count  123.333333      130.0  2022-03-21T14:30:00+08:00
13            cartservice2-0  trace.from_frontend2-0.row_count  100.666667      120.0  2022-03-21T14:30:00+08:00
14  recommendationservice2-0  trace.from_frontend2-0.row_count   81.666667       99.0  2022-03-21T14:30:00+08:00
15         currencyservice-0   trace.from_frontend-1.row_count   75.500000       90.0  2022-03-21T14:30:00+08:00
16         currencyservice-2   trace.from_frontend-1.row_count   75.500000       90.0  2022-03-21T14:30:00+08:00
17   productcatalogservice-1   trace.from_frontend-2.row_count   78.666667       89.0  2022-03-21T14:30:00+08:00
18         currencyservice-1   trace.from_frontend-1.row_count   75.333333       89.0  2022-03-21T14:30:00+08:00
19   productcatalogservice-2   trace.from_frontend-2.row_count   78.666667       89.0  2022-03-21T14:30:00+08:00)```
```

In [ ]:
# In[5]:

# Filter df_stats for the specified candidate components and return a compact table (stored for follow-ups).
try:
    df_stats
except NameError:
    raise RuntimeError("df_stats not found in kernel. Please run the previous aggregation step to produce df_stats.")

candidates = [
    'frontend-0','frontend-1','frontend-2','frontend2-0',
    'node-1','node-2','node-3','node-4','node-5','node-6',
    'adservice','adservice2'
]

# Select and sort as requested
df_candidate_anomalies = df_stats[df_stats['cmdb_id'].isin(candidates)].copy()

cols = ['cmdb_id', 'kpi_name', 'global_p95', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']
# Ensure columns exist
missing = [c for c in cols if c not in df_candidate_anomalies.columns]
if missing:
    raise RuntimeError(f"Missing expected columns in df_stats: {missing}")

df_candidate_anomalies = df_candidate_anomalies.loc[:, cols]

# Sort by cmdb_id (asc), then anomaly_count desc, then max_value_in_window desc
df_candidate_anomalies_sorted = df_candidate_anomalies.sort_values(
    by=['cmdb_id', 'anomaly_count', 'max_value_in_window'],
    ascending=[True, False, False]
)

# Limit to top 40 rows overall and store for follow-ups
df_candidate_anomalies_top40 = df_candidate_anomalies_sorted.head(40).reset_index(drop=True)

# Display the compact table
df_candidate_anomalies_top40

```
Out[5]:
```
```python
# Summarize the df_candidate_anomalies_top40 table (compact, plain English)
try:
    df_candidate_anomalies_top40
except NameError:
    raise RuntimeError("df_candidate_anomalies_top40 not found in kernel.")

import pandas as pd

rows_returned = int(df_candidate_anomalies_top40.shape[0])
comp_counts = df_candidate_anomalies_top40['cmdb_id'].value_counts().to_dict()
top_components = list(comp_counts.items())[:6]  # show up to top 6 components in this slice

# earliest/latest earliest_anomaly_timestamp present in the returned slice
ts_series = pd.to_datetime(df_candidate_anomalies_top40['earliest_anomaly_timestamp'], utc=True, errors='coerce')
if ts_series.dropna().empty:
    earliest_iso = None
    latest_iso = None
else:
    earliest_iso = ts_series.min().tz_convert('Asia/Shanghai').isoformat()
    latest_iso = ts_series.max().tz_convert('Asia/Shanghai').isoformat()

# Example top KPIs for adservice in this returned table
adservice_kpis = df_candidate_anomalies_top40[df_candidate_anomalies_top40['cmdb_id']=='adservice']['kpi_name'].head(5).tolist()

summary_lines = [
    f"Returned rows: {rows_returned} (stored in variable df_candidate_anomalies_top40).",
    "Component counts in this returned slice (component:rows): " + ", ".join([f"{c}:{n}" for c,n in top_components]),
    f"Earliest anomaly timestamp in slice: {earliest_iso}",
    f"Latest anomaly timestamp in slice:   {latest_iso}",
    "",
    "Observed pattern (from the returned rows):",
    f"- Many entries in this top-40 slice are for 'adservice' and show JVM/OS related KPIs (e.g., {', '.join(adservice_kpis)})",
    "- anomaly_count values in the slice are small integers (many = 2) indicating a few points above global P95 within the incident window.",
    "",
    "Notes:",
    "- The full filtered table for follow-ups is in variable `df_candidate_anomalies_top40`.",
    "- You can inspect more rows with df_candidate_anomalies_top40.head(n) or examine df_candidate_anomalies (full filtered set) if needed."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    global_p95  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0   adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  1.019248e+12              2  2022-03-21T14:58:00+08:00         1.019515e+12
1   adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  9.923126e+10              2  2022-03-21T14:57:00+08:00         9.928187e+10
2   adservice  runtime.java_lang_Threading_CurrentThreadUserTime  9.654046e+10              2  2022-03-21T14:57:00+08:00         9.660417e+10
3   adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  7.870851e+10              2  2022-03-21T14:58:00+08:00         7.873011e+10
4   adservice  runtime.java_lang_OperatingSystem_FreePhysical...  4.048402e+08              2  2022-03-21T14:54:00+08:00         4.048445e+08
..        ...                                                ...           ...            ...                        ...                  ...
35  adservice  runtime.java_lang_OperatingSystem_SystemLoadAv...  2.675833e+00              2  2022-03-21T14:39:00+08:00         3.280000e+00
36  adservice                        runtime.jvm_threads_started  7.958333e-01              2  2022-03-21T14:41:00+08:00         9.166667e-01
37  adservice  runtime.java_lang_Memory_ObjectPendingFinaliza...  3.416667e-01              2  2022-03-21T14:45:00+08:00         6.666667e-01
38  adservice     runtime.jvm_memory_pool_MB_used.Survivor_Space  2.106538e-01              2  2022-03-21T14:49:00+08:00         2.365386e-01
39  adservice    runtime.java_lang_OperatingSystem_SystemCpuLoad  1.791901e-02              2  2022-03-21T14:30:00+08:00         1.791977e-02

[40 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Filter df_stats for the requested candidate components and return a compact table.
# Reuse existing df_stats in the kernel.

try:
    df_stats
except NameError:
    raise RuntimeError("df_stats not found in kernel. Run previous steps to produce df_stats before this cell.")

candidates = [
    'frontend-0','frontend-1','frontend-2','frontend2-0',
    'node-1','node-2','node-3','node-4','node-5','node-6',
    'adservice','adservice2'
]

cols_needed = ['cmdb_id', 'kpi_name', 'global_p95', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']
missing = [c for c in cols_needed if c not in df_stats.columns]
if missing:
    raise RuntimeError(f"df_stats is missing required columns: {missing}")

# Filter and sort
df_candidate_details = (
    df_stats[df_stats['cmdb_id'].isin(candidates)]
    .loc[:, cols_needed]
    .sort_values(by=['cmdb_id', 'anomaly_count', 'max_value_in_window'], ascending=[True, False, False])
    .head(40)
    .reset_index(drop=True)
)

# Display the compact table (stored in df_candidate_details for follow-ups)
df_candidate_details

```
Out[6]:
```
```python
# Plain-English summary of df_candidate_details (compact)
summary = (
    "I returned df_candidate_details (40 rows) containing per-(cmdb_id, kpi_name) anomaly stats "
    "for the requested components. The top-40 slice is dominated by 'adservice' JVM/OS-related KPIs "
    "(e.g., ProcessCpuTime, threading CPU/user/alloc metrics, memory and system-load metrics). "
    "For these KPIs anomaly_count is typically small (many = 2), with earliest anomaly timestamps "
    "ranging from ~2022-03-21T14:30:00+08 to ~2022-03-21T14:58:00+08 and max values showing spikes above the global P95. "
    "This table is stored in the kernel variable `df_candidate_details` for follow-ups (inspect it directly for full rows)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    global_p95  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0   adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  1.019248e+12              2  2022-03-21T14:58:00+08:00         1.019515e+12
1   adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  9.923126e+10              2  2022-03-21T14:57:00+08:00         9.928187e+10
2   adservice  runtime.java_lang_Threading_CurrentThreadUserTime  9.654046e+10              2  2022-03-21T14:57:00+08:00         9.660417e+10
3   adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  7.870851e+10              2  2022-03-21T14:58:00+08:00         7.873011e+10
4   adservice  runtime.java_lang_OperatingSystem_FreePhysical...  4.048402e+08              2  2022-03-21T14:54:00+08:00         4.048445e+08
..        ...                                                ...           ...            ...                        ...                  ...
35  adservice  runtime.java_lang_OperatingSystem_SystemLoadAv...  2.675833e+00              2  2022-03-21T14:39:00+08:00         3.280000e+00
36  adservice                        runtime.jvm_threads_started  7.958333e-01              2  2022-03-21T14:41:00+08:00         9.166667e-01
37  adservice  runtime.java_lang_Memory_ObjectPendingFinaliza...  3.416667e-01              2  2022-03-21T14:45:00+08:00         6.666667e-01
38  adservice     runtime.jvm_memory_pool_MB_used.Survivor_Space  2.106538e-01              2  2022-03-21T14:49:00+08:00         2.365386e-01
39  adservice    runtime.java_lang_OperatingSystem_SystemCpuLoad  1.791901e-02              2  2022-03-21T14:30:00+08:00         1.791977e-02

[40 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

# Compute severity scores for selected components and return top 20 by severity.
import numpy as np
import pandas as pd

# Reuse df_stats (precomputed) if available
try:
    df_stats
except NameError:
    raise RuntimeError("df_stats not found in kernel. Please run prior aggregation steps to produce df_stats.")

candidates = [
    'frontend-0','frontend-1','frontend-2','frontend2-0',
    'node-1','node-2','node-3','node-4','node-5','node-6',
    'adservice','adservice2'
]

# Work from df_stats to ensure full coverage
df_work = df_stats[df_stats['cmdb_id'].isin(candidates)].copy()

# Ensure numeric columns
for col in ['global_p95', 'max_value_in_window', 'anomaly_count']:
    if col in df_work.columns:
        df_work[col] = pd.to_numeric(df_work[col], errors='coerce')
    else:
        df_work[col] = np.nan

# Compute severity per rule:
# if global_p95 != 0 then severity = (max_value_in_window - global_p95) / abs(global_p95)
# else severity = max_value_in_window - global_p95
eps = 0.0
is_nonzero = (~df_work['global_p95'].isna()) & (df_work['global_p95'] != 0)
df_work['severity'] = np.where(
    is_nonzero,
    (df_work['max_value_in_window'] - df_work['global_p95']) / df_work['global_p95'].abs(),
    df_work['max_value_in_window'] - df_work['global_p95']
)

# Keep requested columns and sort by severity desc, then cmdb_id asc for tie-breaker
cols_out = ['cmdb_id', 'kpi_name', 'global_p95', 'max_value_in_window', 'anomaly_count', 'earliest_anomaly_timestamp', 'severity']
df_severity_top20 = (
    df_work.loc[:, cols_out]
    .sort_values(by=['severity', 'cmdb_id'], ascending=[False, True])
    .head(20)
    .reset_index(drop=True)
)

# Return the compact table (stored in df_severity_top20 for follow-ups)
df_severity_top20

```
Out[7]:
```
```python
# Plain-English summary of df_severity_top20 (compact)
try:
    df_severity_top20
except NameError:
    raise RuntimeError("df_severity_top20 not found in kernel.")

# Compose a concise summary using top entries from df_severity_top20
top = df_severity_top20.head(10)

summary_lines = [
    "Severity summary (top anomalies among selected components):",
    "",
    "1) Highest-severity findings (examples):",
    f"- {top.iloc[0]['cmdb_id']} | {top.iloc[0]['kpi_name']}  →  severity={top.iloc[0]['severity']:.0f}",
    f"- {top.iloc[1]['cmdb_id']} | {top.iloc[1]['kpi_name']}  →  severity={top.iloc[1]['severity']:.1f}",
    f"- {top.iloc[2]['cmdb_id']} | {top.iloc[2]['kpi_name']}  →  severity={top.iloc[2]['severity']:.1f}",
    "",
    "2) Primary patterns observed:",
    "- The top severity scores are dominated by container CPU throttling metrics on frontend pods (container_cpu_cfs_throttled_*).",
    "- Several entries show very large severity because the global P95 is near zero while a large spike occurred (relative formula amplifies these).",
    "- Other notable anomalies include container memory failure counters (very large absolute spikes) and node I/O spikes (read/write KB/s).",
    "",
    "3) Interpretation / immediate hypothesis:",
    "- Frontend pods experienced extreme CPU throttling (likely CPU quota/cgroup limits or CPU saturation), coinciding with spikes in container I/O and memory-failure metrics.",
    "- The severity metric is relative to global P95; when P95 ≈ 0, even moderate absolute values produce huge severity — inspect raw values (max_value_in_window) as well.",
    "",
    "Variables for follow-up:",
    "- The full top-20 table is in `df_severity_top20` (use this to inspect specific global_p95 and max_value_in_window).",
    ""
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name   global_p95  max_value_in_window  anomaly_count earliest_anomaly_timestamp       severity
0    frontend-0  container.node-5.container_cpu_cfs_throttled_s...     0.000379           322.039851              2  2022-03-21T14:46:00+08:00  850537.587677
1   frontend2-0  container.node-6.container_cpu_cfs_throttled_s...     0.168606           372.655515              2  2022-03-21T14:56:00+08:00    2209.221050
2    frontend-2  container.node-5.container_cpu_cfs_throttled_s...     0.000000           318.103752              1  2022-03-21T14:59:00+08:00     318.103752
3    frontend-2  container.node-5.container_cpu_cfs_throttled_p...     0.000000           315.500000              1  2022-03-21T14:59:00+08:00     315.500000
4    frontend-0  container.node-5.container_cpu_cfs_throttled_p...     0.000000           307.000000              1  2022-03-21T14:59:00+08:00     307.000000
5   frontend2-0       container.node-6.container_fs_reads./dev/vda     0.000000           252.000000              1  2022-03-21T14:59:00+08:00     252.000000
6    frontend-1  container.node-5.container_cpu_cfs_throttled_s...     0.000000           209.772814              1  2022-03-21T14:59:00+08:00     209.772814
7    frontend-1  container.node-5.container_cpu_cfs_throttled_p...     0.000000           206.000000              1  2022-03-21T14:59:00+08:00     206.000000
8        node-6                                    system.io.rkb_s     0.000000           114.000000              1  2022-03-21T14:43:00+08:00     114.000000
9   frontend2-0  container.node-6.container_memory_failures.con...     0.000000           111.000000              1  2022-03-21T14:59:00+08:00     111.000000
10  frontend2-0  container.node-6.container_memory_failures.hie...     0.000000           111.000000              1  2022-03-21T14:59:00+08:00     111.000000
11  frontend2-0  container.node-6.container_cpu_cfs_throttled_p...     3.275000           361.000000              2  2022-03-21T14:56:00+08:00     109.229008
12   frontend-2        container.node-5.container_cpu_user_seconds     0.210500            12.845000              2  2022-03-21T14:42:00+08:00      60.021378
13       node-4                                      system.io.w_s     0.275000            10.000000              2  2022-03-21T14:55:00+08:00      35.363636
14   frontend-2       container.node-5.container_cpu_usage_seconds     0.383415            12.954736              2  2022-03-21T14:35:00+08:00      32.787757
15   frontend-0       container.node-5.container_fs_reads./dev/vda     0.000000            30.000000              1  2022-03-21T14:59:00+08:00      30.000000
16   frontend-1        container.node-5.container_cpu_user_seconds     0.305000             8.450000              1  2022-03-21T14:59:00+08:00      26.704918
17   frontend-0        container.node-5.container_cpu_user_seconds     0.535500            12.415000              2  2022-03-21T14:35:00+08:00      22.183940
18   frontend-2  container.node-5.container_memory_failures.con...  1930.150000         42356.500000              2  2022-03-21T14:39:00+08:00      20.944668
19   frontend-2  container.node-5.container_memory_failures.hie...  1930.150000         42356.500000              2  2022-03-21T14:39:00+08:00      20.944668```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse df_stats from kernel
try:
    df_stats
except NameError:
    raise RuntimeError("df_stats not found in kernel. Run previous steps to produce df_stats.")

tz = pytz.timezone('Asia/Shanghai')

# Candidate components and early window (inclusive)
candidates = [
    'frontend-0','frontend-1','frontend-2','frontend2-0',
    'node-1','node-2','node-3','node-4','node-5','node-6'
]
start_local = tz.localize(datetime(2022, 3, 21, 14, 30, 0))
end_local   = tz.localize(datetime(2022, 3, 21, 14, 33, 0))

# Parse earliest_anomaly_timestamp into timezone-aware datetimes (coerce errors)
earliest_dt = pd.to_datetime(df_stats['earliest_anomaly_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)

# Build working DataFrame with parsed earliest_dt
df_work = df_stats.copy()
df_work['_earliest_dt'] = earliest_dt

# Filter by candidates and earliest_anomaly_timestamp within [start_local, end_local]
mask_comp = df_work['cmdb_id'].isin(candidates)
mask_time = df_work['_earliest_dt'].notna() & (df_work['_earliest_dt'] >= start_local) & (df_work['_earliest_dt'] <= end_local)
df_early = df_work[mask_comp & mask_time].copy()

# Ensure numeric columns exist
for c in ['global_p95', 'max_value_in_window', 'anomaly_count']:
    if c in df_early.columns:
        df_early[c] = pd.to_numeric(df_early[c], errors='coerce')
    else:
        df_early[c] = np.nan

# Compute severity per spec
is_nonzero = (~df_early['global_p95'].isna()) & (df_early['global_p95'] != 0)
df_early['severity'] = np.where(
    is_nonzero,
    (df_early['max_value_in_window'] - df_early['global_p95']) / df_early['global_p95'].abs(),
    df_early['max_value_in_window'] - df_early['global_p95']
)

# Prepare df_early_severity: select columns and format earliest timestamp as ISO (Asia/Shanghai)
df_early['earliest_anomaly_timestamp'] = df_early['_earliest_dt'].apply(lambda x: None if pd.isna(x) else x.isoformat())
cols_out = ['cmdb_id', 'kpi_name', 'global_p95', 'max_value_in_window', 'anomaly_count', 'earliest_anomaly_timestamp', 'severity']
df_early_severity = df_early.loc[:, cols_out].sort_values(by=['severity'], ascending=False).head(20).reset_index(drop=True)

# Component-level aggregate for the filtered set
grouped = df_early.groupby('cmdb_id', sort=True)
agg_rows = []
for cmdb, g in grouped:
    total_anomaly_points = int(g['anomaly_count'].sum(min_count=1)) if not g['anomaly_count'].isna().all() else 0
    num_kpis_with_anom = int((g['anomaly_count'] > 0).sum())
    # earliest across kpis (use parsed _earliest_dt)
    if g['_earliest_dt'].dropna().empty:
        earliest_iso = None
    else:
        earliest_iso = g['_earliest_dt'].dropna().min().isoformat()
    max_severity = float(g['severity'].max()) if not g['severity'].isna().all() else np.nan
    agg_rows.append((cmdb, total_anomaly_points, num_kpis_with_anom, earliest_iso, max_severity))

df_early_component_agg = pd.DataFrame(
    agg_rows,
    columns=['cmdb_id', 'total_anomaly_points', 'num_kpis_with_anom', 'earliest_anomaly_timestamp', 'max_severity']
)

# Sort component aggregate by total_anomaly_points desc for compactness (<=10 rows here)
df_early_component_agg = df_early_component_agg.sort_values(by=['total_anomaly_points', 'max_severity'], ascending=[False, False]).reset_index(drop=True)

# Store variables in kernel for follow-ups (names as requested)
# df_early_severity, df_early_component_agg

df_early_severity, df_early_component_agg

```
Out[8]:
```
```python
# Summarize the early-incident (14:30–14:33 UTC+8) findings in plain English.
# Reuse existing kernel variables: df_early_severity, df_early_component_agg

try:
    df_early_severity
    df_early_component_agg
except NameError:
    raise RuntimeError("Expected df_early_severity and df_early_component_agg in kernel.")

# Extract compact highlights
n_kpis = int(df_early_severity.shape[0])
components = df_early_component_agg['cmdb_id'].tolist()
top_component = df_early_component_agg.iloc[0].to_dict() if not df_early_component_agg.empty else None

summary_lines = []

summary_lines.append("Early-incident summary (window 2022-03-21 14:30–14:33 UTC+8):")

# Overall component-level takeaway
if top_component:
    summary_lines.append(f"- {len(components)} components had early anomalies; the top component by total anomaly points is "
                         f"{top_component['cmdb_id']} (total_anomaly_points={int(top_component['total_anomaly_points'])}, "
                         f"num_kpis_with_anom={int(top_component['num_kpis_with_anom'])}, earliest={top_component['earliest_anomaly_timestamp']}).")
else:
    summary_lines.append("- No early anomalies found in the selected components.")

# Key observations from KPI-level severity
if n_kpis > 0:
    # Top severity row
    top_row = df_early_severity.iloc[0]
    summary_lines.append(f"- The single highest-severity KPI is on {top_row['cmdb_id']}: {top_row['kpi_name']} "
                         f"(earliest anomaly {top_row['earliest_anomaly_timestamp']}, severity={top_row['severity']:.3f}, "
                         f"max={top_row['max_value_in_window']}, global_P95={top_row['global_p95']}).")
    # Patterns
    summary_lines.append("- Major patterns observed:")
    summary_lines.append("  * Frontend pods (notably frontend-1) show very large memory-failure spikes around 14:32, giving very high severity scores.")
    summary_lines.append("  * Node-level I/O and queue-length metrics (especially on node-2 and node-3) spiked at ~14:30 and show high relative severity.")
    summary_lines.append("  * Several mesh/request metrics for frontend sources increased around 14:31–14:32, indicating elevated request/activity in that early window.")
    summary_lines.append("  * Severity is relative to global P95; very large ratios often stem from near-zero P95 plus a large absolute spike—so inspect both severity and absolute max values.")
else:
    summary_lines.append("- No KPI anomalies matched the early-time filter.")

summary_lines.append("")
summary_lines.append("Stored variables for follow-up: df_early_severity (KPI-level rows) and df_early_component_agg (component aggregates).")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name     global_p95  max_value_in_window  anomaly_count earliest_anomaly_timestamp   severity
0   frontend-1  container.node-5.container_memory_failures.hie...    2864.525000         42569.500000              2  2022-03-21T14:32:00+08:00  13.860928
1   frontend-1  container.node-5.container_memory_failures.con...    2864.525000         42569.500000              2  2022-03-21T14:32:00+08:00  13.860928
2       node-2                                 system.io.avg_q_sz       0.045500             0.500000              2  2022-03-21T14:30:00+08:00   9.989011
3   frontend-1  mesh.destination.unknown.frontend.istio_reques...       0.080260             0.486675              2  2022-03-21T14:31:00+08:00   5.063730
4   frontend-0  mesh.source.frontend.jaeger-collector.istio_re...       0.016388             0.081225              2  2022-03-21T14:32:00+08:00   3.956522
5       node-2                                      system.io.w_s      27.525000           100.500000              2  2022-03-21T14:30:00+08:00   2.651226
6       node-2                                  system.io.w_await       0.721000             2.530000              2  2022-03-21T14:30:00+08:00   2.509015
7       node-2                                    system.io.await       0.721000             2.530000              2  2022-03-21T14:30:00+08:00   2.509015
8       node-5                                     system.io.util       0.577500             1.850000              2  2022-03-21T14:32:00+08:00   2.203463
9       node-4                        system.net.tcp.retrans_segs       0.141000             0.380000              2  2022-03-21T14:31:00+08:00   1.695035
10      node-5                                    system.cpu.user       4.795000            11.450000              2  2022-03-21T14:31:00+08:00   1.387904
11      node-5                                    system.io.svctm       0.155500             0.350000              2  2022-03-21T14:32:00+08:00   1.250804
12      node-3                            system.net.tcp.out_segs    1731.558500          3256.290000              2  2022-03-21T14:31:00+08:00   0.880554
13      node-5                               system.cpu.pct_usage       7.760500            14.060000              2  2022-03-21T14:31:00+08:00   0.811739
14      node-2                            system.net.tcp.out_segs    2071.055500          3740.610000              2  2022-03-21T14:32:00+08:00   0.806137
15      node-3                             system.net.tcp.in_segs     559.991500           802.390000              2  2022-03-21T14:31:00+08:00   0.432861
16      node-2                              system.net.bytes_sent  294787.457000        422253.500000              2  2022-03-21T14:32:00+08:00   0.432400
17  frontend-1  mesh.source.frontend.currencyservice.istio_req...       0.253296             0.338100              2  2022-03-21T14:32:00+08:00   0.334801
18      node-3                              system.net.bytes_sent  509057.747500        649421.900000              2  2022-03-21T14:31:00+08:00   0.275733
19      node-2                        system.net.packets_in.count     171.273000           217.220000              2  2022-03-21T14:32:00+08:00   0.268268,        cmdb_id  total_anomaly_points  num_kpis_with_anom earliest_anomaly_timestamp  max_severity
0   frontend-1                    30                  15  2022-03-21T14:31:00+08:00     13.860928
1       node-2                    27                  14  2022-03-21T14:30:00+08:00      9.989011
2       node-3                    23                  12  2022-03-21T14:30:00+08:00      0.880554
3  frontend2-0                    23                  13  2022-03-21T14:30:00+08:00      0.059603
4       node-5                    17                   9  2022-03-21T14:30:00+08:00      2.203463
5       node-6                    11                   6  2022-03-21T14:30:00+08:00      0.208791
6   frontend-0                    10                   5  2022-03-21T14:31:00+08:00      3.956522
7       node-4                     8                   4  2022-03-21T14:30:00+08:00      1.695035
8   frontend-2                     6                   3  2022-03-21T14:32:00+08:00      0.017565
9       node-1                     6                   3  2022-03-21T14:30:00+08:00      0.000085)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse kernel variables
try:
    df_early_severity
except NameError:
    raise RuntimeError("df_early_severity not found in kernel. Please run the early-incident extraction step first.")

try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Window for sample extraction (inclusive)
start_local = tz.localize(datetime(2022, 3, 21, 14, 30, 0))
end_local   = tz.localize(datetime(2022, 3, 21, 14, 35, 0))
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts = int(end_local.astimezone(pytz.utc).timestamp())

# 1) Select KPI rows for frontend-1 from df_early_severity (limit to 10 KPIs)
df_frontend1_early = df_early_severity[df_early_severity['cmdb_id'] == 'frontend-1'].copy()
kpi_list = df_frontend1_early['kpi_name'].dropna().unique().tolist()[:10]

# 2) Prepare KPI-level stats table
rows = []
for kpi in kpi_list:
    row = df_frontend1_early[df_frontend1_early['kpi_name'] == kpi].iloc[0]
    rows.append({
        'kpi_name': row['kpi_name'],
        'global_p95': row['global_p95'],
        'anomaly_count': int(row['anomaly_count']),
        'earliest_anomaly_timestamp': row['earliest_anomaly_timestamp'],
        'max_value_in_window': row['max_value_in_window']
    })

df_frontend1_kpi_stats = pd.DataFrame(rows)
if df_frontend1_kpi_stats.empty:
    df_frontend1_kpi_stats = pd.DataFrame(columns=['kpi_name','global_p95','anomaly_count','earliest_anomaly_timestamp','max_value_in_window'])
else:
    df_frontend1_kpi_stats = df_frontend1_kpi_stats.sort_values(['anomaly_count','kpi_name'], ascending=[False, True]).reset_index(drop=True)

# 3) For each KPI, get up to first 20 metric rows within window where value > global_p95
frontend1_kpi_samples = {}
for _, r in df_frontend1_kpi_stats.iterrows():
    kpi = r['kpi_name']
    gp95 = r['global_p95']
    if pd.isna(gp95):
        df_samples = pd.DataFrame(columns=['timestamp', 'value'])
    else:
        df_s = df_metric[
            (df_metric['timestamp'] >= start_ts) &
            (df_metric['timestamp'] <= end_ts) &
            (df_metric['cmdb_id'] == 'frontend-1') &
            (df_metric['kpi_name'] == kpi) &
            (df_metric['value'] > gp95)
        ].copy()
        if not df_s.empty:
            df_s['timestamp'] = pd.to_datetime(df_s['timestamp'], unit='s', utc=True).dt.tz_convert(tz).apply(lambda x: x.isoformat())
            df_samples = df_s[['timestamp', 'value']].head(20).reset_index(drop=True)
        else:
            df_samples = pd.DataFrame(columns=['timestamp', 'value'])
    frontend1_kpi_samples[kpi] = df_samples

# Store for follow-ups in kernel variables
# df_frontend1_kpi_stats (DataFrame) and frontend1_kpi_samples (dict of DataFrames)

df_frontend1_kpi_stats

```
Out[9]:
```
```python
# Plain-English summary of frontend-1 early-incident KPI findings (compact).
summary = (
    "Frontend-1 early-incident summary (14:30–14:33 UTC+8):\n"
    "- KPIs flagged (up to 10 shown):\n"
    "  1) container.node-5.container_memory_failures.count  — global_P95=2864.525, anomalies=2, earliest=2022-03-21T14:32:00+08:00, max=42569.5\n"
    "  2) container.node-5.container_memory_failures.hierarchy? — global_P95=2864.525, anomalies=2, earliest=2022-03-21T14:32:00+08:00, max=42569.5\n"
    "     (Both memory-failure metrics show a very large absolute spike at ~14:32 — strong signal of memory-related failures on frontend-1.)\n"
    "  3) mesh.destination.unknown.frontend.istio_request... — global_P95≈0.08026, anomalies=2, earliest=2022-03-21T14:31:00+08:00, max≈0.486675\n"
    "  4) mesh.source.frontend.currencyservice.istio_requests... — global_P95≈0.25330, anomalies=2, earliest=2022-03-21T14:32:00+08:00, max≈0.33810\n"
    "     (Mesh/request metrics increased around 14:31–14:32, indicating elevated request activity/mesh traffic.)\n"
    "- Interpretation: frontend-1 exhibits pronounced memory-failure spikes at ~14:32 (very large absolute values far above P95),"
    " accompanied by increased mesh request metrics ~14:31–14:32. This suggests a memory-related failure or crash activity on frontend-1 coincident with elevated request activity.\n"
    "- Follow-ups: detailed anomaly rows (timestamp/value samples) for each KPI are stored in variable `frontend1_kpi_samples`; KPI-level stats are in `df_frontend1_kpi_stats`."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name   global_p95  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0  container.node-5.container_memory_failures.con...  2864.525000              2  2022-03-21T14:32:00+08:00         42569.500000
1  container.node-5.container_memory_failures.hie...  2864.525000              2  2022-03-21T14:32:00+08:00         42569.500000
2  mesh.destination.unknown.frontend.istio_reques...     0.080260              2  2022-03-21T14:31:00+08:00             0.486675
3  mesh.source.frontend.currencyservice.istio_req...     0.253296              2  2022-03-21T14:32:00+08:00             0.338100```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse kernel variables
try:
    df_stats
except NameError:
    raise RuntimeError("df_stats not found in kernel. Run prior steps to produce df_stats.")

try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# 1) Filter df_stats for adservice/adservice2 and earliest_anomaly_timestamp in [14:55,15:00] UTC+8
components = ['adservice', 'adservice2']
start_local = tz.localize(datetime(2022, 3, 21, 14, 55, 0))
end_local   = tz.localize(datetime(2022, 3, 21, 15, 0, 0))

# Parse earliest_anomaly_timestamp to timezone-aware datetimes (coerce missing)
ea = pd.to_datetime(df_stats['earliest_anomaly_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
df_stats_work = df_stats.copy()
df_stats_work['_earliest_dt'] = ea

mask_comp = df_stats_work['cmdb_id'].isin(components)
mask_time = df_stats_work['_earliest_dt'].notna() & (df_stats_work['_earliest_dt'] >= start_local) & (df_stats_work['_earliest_dt'] <= end_local)
df_ad = df_stats_work[mask_comp & mask_time].copy()

# Ensure numeric columns
for c in ['global_p95', 'max_value_in_window', 'anomaly_count']:
    df_ad[c] = pd.to_numeric(df_ad.get(c, pd.Series(np.nan, index=df_ad.index)), errors='coerce')

# Compute severity per rule
is_nonzero = (~df_ad['global_p95'].isna()) & (df_ad['global_p95'] != 0)
df_ad['severity'] = np.where(
    is_nonzero,
    (df_ad['max_value_in_window'] - df_ad['global_p95']) / df_ad['global_p95'].abs(),
    df_ad['max_value_in_window'] - df_ad['global_p95']
)

# Prepare df_adservice_kpi_stats (limit top 20 by severity desc)
cols_out = ['cmdb_id', 'kpi_name', 'global_p95', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'severity']
# Ensure earliest_anomaly_timestamp column exists (use string if present, else derive from _earliest_dt)
if 'earliest_anomaly_timestamp' not in df_ad.columns:
    df_ad['earliest_anomaly_timestamp'] = df_ad['_earliest_dt'].apply(lambda x: None if pd.isna(x) else x.isoformat())
df_adservice_kpi_stats = df_ad.loc[:, cols_out].sort_values(by='severity', ascending=False).head(20).reset_index(drop=True)

# Store in kernel variable as required
df_adservice_kpi_stats = df_adservice_kpi_stats

# 2) For top 5 KPI rows by severity, extract metric datapoints in window 14:55–15:00 UTC+8 with value > global_p95
top5 = df_adservice_kpi_stats.head(5).copy()

# Convert window to epoch seconds (UTC) for filtering df_metric (timestamps in seconds UTC floored)
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts   = int(end_local.astimezone(pytz.utc).timestamp())

adservice_kpi_samples = {}
for _, r in top5.iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    gp95 = r['global_p95']
    # If GP95 is NaN, skip and return empty DataFrame
    if pd.isna(gp95):
        df_samp = pd.DataFrame(columns=['timestamp', 'value'])
    else:
        df_s = df_metric[
            (df_metric['timestamp'] >= start_ts) &
            (df_metric['timestamp'] <= end_ts) &
            (df_metric['cmdb_id'] == cmdb) &
            (df_metric['kpi_name'] == kpi) &
            (df_metric['value'] > gp95)
        ].copy()
        if not df_s.empty:
            # Convert timestamp to ISO in Asia/Shanghai for readability
            df_s['timestamp'] = pd.to_datetime(df_s['timestamp'], unit='s', utc=True).dt.tz_convert(tz).apply(lambda x: x.isoformat())
            df_samp = df_s[['timestamp', 'value']].head(20).reset_index(drop=True)
        else:
            df_samp = pd.DataFrame(columns=['timestamp', 'value'])
    # store by a safe key (cmdb|kpi short)
    key = f"{cmdb} | {kpi}"
    adservice_kpi_samples[key] = df_samp

# Expose the results (compact): df_adservice_kpi_stats and adservice_kpi_samples
df_adservice_kpi_stats, adservice_kpi_samples

```
Out[10]:
```
```python
# Compose a concise plain-English summary of the adservice/adservice2 late-incident validation
# using precomputed variables df_adservice_kpi_stats and adservice_kpi_samples from the kernel.

try:
    df_adservice_kpi_stats
    adservice_kpi_samples
except NameError:
    raise RuntimeError("Expected df_adservice_kpi_stats and adservice_kpi_samples in kernel.")

# Build summary lines
lines = []
lines.append("Adservice late-incident validation (14:55–15:00 UTC+8) — compact summary:")

# high-level counts
n_kpis = int(df_adservice_kpi_stats.shape[0])
lines.append(f"- KPIs matched: {n_kpis} (filtered by earliest anomaly timestamp in 14:55–15:00).")

# Top KPIs by severity (show up to 5)
top = df_adservice_kpi_stats.head(5)
lines.append("- Top KPI breaches (by severity):")
for i, row in top.reset_index(drop=True).iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    gp95 = row['global_p95']
    cnt = int(row['anomaly_count']) if not pd.isna(row['anomaly_count']) else "N/A"
    ets = row['earliest_anomaly_timestamp']
    maxv = row['max_value_in_window']
    sev = row['severity']
    # sample count available in adservice_kpi_samples
    key = f"{cmdb} | {kpi}"
    samp = adservice_kpi_samples.get(key, pd.DataFrame())
    samp_count = int(samp.shape[0])
    lines.append(f"  {i+1}. {cmdb} | {kpi} — global_P95={gp95}, anomalies={cnt}, earliest={ets}, max={maxv}, severity={sev:.6f}; sample_rows={samp_count}")

# Notable interpretation based strictly on data
lines.append("- Interpretation (data-driven):")
lines.append("  * adservice and adservice2 show small relative severity values for many JVM/OS KPIs (severity ~0.001–0.03), indicating modest breaches above their global P95 in this late window.")
lines.append("  * The clearest actionable breach is adservice | app.grpc.count (global_P95≈252.3) which recorded metric samples exceeding P95 (e.g., 255 at 14:56 and 260 at 14:58).")
lines.append("  * adservice2 shows multiple JVM GC / memory/thread metrics with small relative increases at ~14:58 — these are consistent JVM activity spikes rather than extreme ratios.")
lines.append("  * Overall: absolute spikes are present but relative severities are modest for most adservice/adservice2 KPIs in this late window; app.grpc.count is a clear, observable breach in metrics samples.")
lines.append("")
lines.append("Stored outputs for follow-up:")
lines.append("- Per-KPI table: df_adservice_kpi_stats")
lines.append("- Per-KPI metric samples (top-5): adservice_kpi_samples")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name    global_p95  anomaly_count earliest_anomaly_timestamp  max_value_in_window  severity
0    adservice                                     app.grpc.count  2.523000e+02              2  2022-03-21T14:56:00+08:00         2.600000e+02  0.030519
1    adservice                 runtime.jvm_threads_state.RUNNABLE  5.000000e+00              1  2022-03-21T14:57:00+08:00         5.083333e+00  0.016667
2   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.341527e+07              2  2022-03-21T14:58:00+08:00         2.350308e+07  0.003750
3   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.341527e+07              2  2022-03-21T14:58:00+08:00         2.350309e+07  0.003750
4   adservice2                   runtime.java_lang_Runtime_Uptime  2.342383e+07              2  2022-03-21T14:58:00+08:00         2.351083e+07  0.003714
5   adservice2  runtime.jvm_memory_pool_allocated_MB_total.Ede...  2.834028e+04              2  2022-03-21T14:58:00+08:00         2.844408e+04  0.003663
6   adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...  5.716434e+09              2  2022-03-21T14:58:00+08:00         5.737365e+09  0.003662
7   adservice2  runtime.java_lang_Threading_TotalStartedThread...  1.410000e+02              1  2022-03-21T14:59:00+08:00         1.415000e+02  0.003546
8   adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime  8.303044e+09              2  2022-03-21T14:58:00+08:00         8.324734e+09  0.002612
9   adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime  1.254475e+11              2  2022-03-21T14:58:00+08:00         1.257608e+11  0.002498
10  adservice2  runtime.java_lang_Threading_CurrentThreadUserTime  7.742708e+09              2  2022-03-21T14:58:00+08:00         7.760833e+09  0.002341
11  adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.684254e+03              2  2022-03-21T14:58:00+08:00         2.690333e+03  0.002265
12  adservice2  runtime.java_lang_GarbageCollector_CollectionC...  2.684254e+03              2  2022-03-21T14:58:00+08:00         2.690333e+03  0.002265
13  adservice2  runtime.java_lang_GarbageCollector_CollectionT...  5.423142e+03              2  2022-03-21T14:58:00+08:00         5.435167e+03  0.002217
14  adservice2  runtime.jvm_memory_pool_allocated_MB_total.Sur...  1.433723e+02              2  2022-03-21T14:58:00+08:00         1.436735e+02  0.002101
15  adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...  3.045735e+07              2  2022-03-21T14:58:00+08:00         3.050799e+07  0.001663
16  adservice2  runtime.java_lang_MemoryPool_Usage_used.Tenure...  3.045735e+07              2  2022-03-21T14:58:00+08:00         3.050798e+07  0.001662
17  adservice2  runtime.java_lang_MemoryPool_PeakUsage_used.Te...  3.045735e+07              2  2022-03-21T14:58:00+08:00         3.050798e+07  0.001662
18  adservice2        runtime.jvm_memory_pool_MB_used.Tenured_Gen  2.904645e+01              2  2022-03-21T14:58:00+08:00         2.909469e+01  0.001661
19   adservice  runtime.java_lang_Threading_TotalStartedThread...  6.252292e+02              2  2022-03-21T14:58:00+08:00         6.260000e+02  0.001233, {'adservice | app.grpc.count':                    timestamp  value
0  2022-03-21T14:56:00+08:00  255.0
1  2022-03-21T14:58:00+08:00  260.0, 'adservice | runtime.jvm_threads_state.RUNNABLE':                    timestamp     value
0  2022-03-21T14:57:00+08:00  5.083333, 'adservice2 | runtime.java_lang_GarbageCollector_LastGcInfo_startTime.Copy':                    timestamp        value
0  2022-03-21T14:58:00+08:00  23442628.75
1  2022-03-21T14:59:00+08:00  23503084.00, 'adservice2 | runtime.java_lang_GarbageCollector_LastGcInfo_endTime.Copy':                    timestamp        value
0  2022-03-21T14:58:00+08:00  23442630.75
1  2022-03-21T14:59:00+08:00  23503086.00, 'adservice2 | runtime.java_lang_Runtime_Uptime':                    timestamp       value
0  2022-03-21T14:58:00+08:00  23450826.0
1  2022-03-21T14:59:00+08:00  23510826.0})```
```